# 1.5 CNN from scratch with a grid head

Trains a CNN from scratch on the preprocessed wrist X-rays. The model takes a 512x256 image and predicts, per 16x16 grid cell, fracture-or-not and four box distances. No pretrained weights.

Utilizes outputs from `0.1_data-pipeline_fahim.ipynb` and `0.2_data-pipeline-localization_fahim.ipynb`. Runs on GPU if available.

| Output (per run folder) | Contents |
|---|---|
| `best_val_auc_model.pt` | model weights at the epoch with the highest val cell AUC |
| `checkpoint_epoch_XX.pt` | resumable training state every 5 epochs |
| `training_history.csv` | both loss parts and val cell AUC per epoch |
| `val_cell_predictions.npz`, `test_cell_predictions.npz` | per-cell scores and distances, read by Fahim's scoring notebook |

Three train sets, one seed to start: `all`, `clean`, `all-castpairs`.

## 0. Imports, settings and data check

Initialize settings and assert that data pipeline notebooks `0.1` and `0.2` have been run on this machine.

In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import roc_auc_score
from torch.utils.data import ConcatDataset, DataLoader, Dataset

from snap.config import MODELS_DIR, PROCESSED_DATA_DIR

# Training settings
SEEDS = [42]              # add more seeds if a run is quick
MAX_EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 10.0
BOX_LOSS_WEIGHT = 1.0
CELL_SIZE = 16

# Paths
NOTEBOOK_DIR = MODELS_DIR / '1.5_small-cnn-grid_lloyd'
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Run folder: {NOTEBOOK_DIR}")

# Check that 0.1 and 0.2 have been run on this machine
required = [
    PROCESSED_DATA_DIR / 'graz.csv',
    PROCESSED_DATA_DIR / 'graz_256x512_u8.npy',
    PROCESSED_DATA_DIR / 'graz_boxes.csv',
    PROCESSED_DATA_DIR / 'graz_cell_targets.npz',
    PROCESSED_DATA_DIR / 'graz_cast_pairs.csv',
    PROCESSED_DATA_DIR / 'graz_cast_pairs_256x512_u8.npy',
    PROCESSED_DATA_DIR / 'graz_cast_pairs_cell_targets.npz',
]
missing = [p for p in required if not p.exists()]
assert not missing, (
    f"Missing files: {missing}. Run 0.1_data-pipeline_fahim and "
    f"0.2_data-pipeline-localization_fahim first."
)
print('All required files present.')

2026-11-23 01:38:07.164 +00:00 | INFO     | snap.config:<module>:11 - PROJ_ROOT path is: /work


Device: cpu
Run folder: /work/models/1.5_small-cnn-grid_lloyd
All required files present.


## 1. Dataset

`GridDataset` returns `(image, objectness, distances)` for one row. The image is normalized with mean and std computed from 2,000 random **original** train images; the extras don't change those numbers.

The same class handles both the originals and the extra cast-pair images; `ConcatDataset` joins them for the `all-castpairs` run.

In [2]:
# Load the split table and both image arrays
df = pd.read_csv(PROCESSED_DATA_DIR / 'graz.csv')
images_orig = np.load(PROCESSED_DATA_DIR / 'graz_256x512_u8.npy', mmap_mode='r')
images_extra = np.load(PROCESSED_DATA_DIR / 'graz_cast_pairs_256x512_u8.npy', mmap_mode='r')

targets_orig = np.load(PROCESSED_DATA_DIR / 'graz_cell_targets.npz')
targets_extra = np.load(PROCESSED_DATA_DIR / 'graz_cast_pairs_cell_targets.npz')

obj_orig = targets_orig['objectness']        # (n, 32, 16) int8
dist_orig = targets_orig['distances']        # (n, 32, 16, 4) float32
obj_extra = targets_extra['objectness']
dist_extra = targets_extra['distances']

# Split masks for the originals
train_mask = (df['split'] == 'train').to_numpy()
val_mask = (df['split'] == 'val').to_numpy()
test_mask = (df['split'] == 'test').to_numpy()
clean_mask = train_mask & (df['clean'] == 1).to_numpy()

train_rows = np.where(train_mask)[0]
val_rows = np.where(val_mask)[0]
test_rows = np.where(test_mask)[0]
clean_rows = np.where(clean_mask)[0]

print(f"Train rows: {len(train_rows)}  (clean: {len(clean_rows)})")
print(f"Val rows:   {len(val_rows)}")
print(f"Test rows:  {len(test_rows)}")
print(f"Extra rows: {len(images_extra)}")

# Normalization constants from 2,000 random original train images
rng_norm = np.random.default_rng(0)
sample_rows = rng_norm.choice(train_rows, size=2000, replace=False)
sample = images_orig[sample_rows].astype(np.float32)
MEAN = float(sample.mean())
STD = float(sample.std())
del sample

print(f"MEAN = {MEAN:.4f}")
print(f"STD  = {STD:.4f}")

class GridDataset(Dataset):
    # images: (n, 512, 256) uint8
    # objectness: (n, 32, 16) int8/int64
    # distances: (n, 32, 16, 4) float32
    # rows: int array of indices into images/objectness/distances
    def __init__(self, images, objectness, distances, rows):
        self.images = images
        self.objectness = objectness
        self.distances = distances
        self.rows = np.asarray(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = int(self.rows[i])
        # Image: (1, 512, 256) float32, normalized with the shared MEAN/STD
        img = self.images[r].astype(np.float32)
        img = (img - MEAN) / STD
        img = torch.from_numpy(img).unsqueeze(0)
        # Targets
        obj = torch.from_numpy(np.array(self.objectness[r])).long()      # (32, 16)
        dist = torch.from_numpy(np.array(self.distances[r])).float()     # (32, 16, 4)
        return img, obj, dist

# Build the datasets. Val and test use original images only.
val_ds = GridDataset(images_orig, obj_orig, dist_orig, val_rows)
test_ds = GridDataset(images_orig, obj_orig, dist_orig, test_rows)

# Train sets
train_sets = {
    'all': GridDataset(images_orig, obj_orig, dist_orig, train_rows),
    'clean': GridDataset(images_orig, obj_orig, dist_orig, clean_rows),
    'all-castpairs': ConcatDataset([
        GridDataset(images_orig, obj_orig, dist_orig, train_rows),
        GridDataset(images_extra, obj_extra, dist_extra, np.arange(len(images_extra))),
    ]),
}

for name, ds in train_sets.items():
    print(f"  {name}: {len(ds)} images")
print(f"  val: {len(val_ds)} images")
print(f"  test: {len(test_ds)} images")

# Smoke test: one batch, shapes and value ranges
smoke_loader = DataLoader(train_sets['all'], batch_size=4, shuffle=False)
img, obj, dist = next(iter(smoke_loader))
print(f"img   {tuple(img.shape)}   dtype={img.dtype}   range=[{img.min():.2f}, {img.max():.2f}]")
print(f"obj   {tuple(obj.shape)}   values={torch.unique(obj).tolist()}")
print(f"dist  {tuple(dist.shape)}  max={dist.max().item():.1f}")

Train rows: 14182  (clean: 7589)
Val rows:   3202
Test rows:  2943
Extra rows: 7018
MEAN = 84.0853
STD  = 69.9201
  all: 14182 images
  clean: 7589 images
  all-castpairs: 21200 images
  val: 3202 images
  test: 2943 images
img   (4, 1, 512, 256)   dtype=torch.float32   range=[-1.20, 2.44]
obj   (4, 32, 16)   values=[0]
dist  (4, 32, 16, 4)  max=0.0


## 2. Model

Five conv blocks. Blocks 1-4 halve the spatial size with a MaxPool2d(2), so 512x256 goes 256x128 -> 128x64 -> 64x32 -> 32x16. Block 5 has no pool, so the map stays 32x16. One pixel of that map is one 16-px grid cell.

Two 1x1 heads on the 32x16x256 map:
- `objectness_head` -> `(batch, 1, 32, 16)` logits
- `box_head` -> `(batch, 4, 32, 16)` predicting `log(distance / CELL_SIZE)`

`forward` returns `(objectness_logits (batch, 32, 16), box_out (batch, 32, 16, 4))`.


In [3]:
def conv_block(in_channels, out_channels, pool):
    layers = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )
    if pool:
        layers.append(nn.MaxPool2d(2))
    return layers

class SmallCNNGrid(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: (batch, 1, 512, 256)
        self.block1 = conv_block(1, 16, pool=True)     # -> (batch, 16, 256, 128)
        self.block2 = conv_block(16, 32, pool=True)    # -> (batch, 32, 128, 64)
        self.block3 = conv_block(32, 64, pool=True)    # -> (batch, 64, 64, 32)
        self.block4 = conv_block(64, 128, pool=True)   # -> (batch, 128, 32, 16)
        self.block5 = conv_block(128, 256, pool=False) # -> (batch, 256, 32, 16)
        self.objectness_head = nn.Conv2d(256, 1, kernel_size=1)  # -> (batch, 1, 32, 16)
        self.box_head = nn.Conv2d(256, 4, kernel_size=1)         # -> (batch, 4, 32, 16)

    def forward(self, x):
        x = self.block1(x)          # (batch, 16, 256, 128)
        x = self.block2(x)          # (batch, 32, 128, 64)
        x = self.block3(x)          # (batch, 64, 64, 32)
        x = self.block4(x)          # (batch, 128, 32, 16)
        x = self.block5(x)          # (batch, 256, 32, 16)
        obj_logits = self.objectness_head(x).squeeze(1)          # (batch, 32, 16)
        box_out = self.box_head(x).permute(0, 2, 3, 1).contiguous()  # (batch, 32, 16, 4)
        return obj_logits, box_out

# Show parameter count
_model = SmallCNNGrid()
n_params = sum(p.numel() for p in _model.parameters())
print(f"Parameters: {n_params:,}")

# Shape check
with torch.no_grad():
    dummy = torch.zeros(2, 1, 512, 256)
    logits, boxes = _model(dummy)
    print(f"logits {tuple(logits.shape)}  boxes {tuple(boxes.shape)}")
del _model

Parameters: 394,597
logits (2, 32, 16)  boxes (2, 32, 16, 4)


## 3. Loss

The objectness loss is a weighted binary cross-entropy that ignores `-1` cells and up-weights the ~1.6 % of cells that are positive. The box loss is an L1 on positive cells only, in log-of-distance space.

In [ ]:
def grid_loss(obj_logits, box_out, objectness, distances):
    # obj_logits: (batch, 32, 16) float
    # box_out:    (batch, 32, 16, 4) float, reads as log(distance / CELL_SIZE)
    # objectness: (batch, 32, 16) long, values in {-1, 0, 1}
    # distances:  (batch, 32, 16, 4) float, pixels
    valid = objectness != -1                # cells counted in the objectness loss
    positive = objectness == 1              # cells counted in the box loss

    # Objectness: weighted BCE, averaged over valid cells only
    pos_weight = torch.tensor(POS_WEIGHT, device=obj_logits.device)
    bce = nn.functional.binary_cross_entropy_with_logits(
        obj_logits,
        positive.float(),
        reduction='none',
        pos_weight=pos_weight,
    )
    objectness_loss = (bce * valid).sum() / valid.sum().clamp(min=1)

    # Box loss: L1 in log(distance / 16) space, positive cells only
    if positive.any():
        target = torch.log(distances.clamp(min=1.0) / CELL_SIZE)
        box_loss = nn.functional.l1_loss(box_out[positive], target[positive])
    else:
        box_loss = torch.zeros((), device=obj_logits.device)

    loss = objectness_loss + BOX_LOSS_WEIGHT * box_loss
    return loss, objectness_loss.detach(), box_loss.detach()

### Smoke test: 64 images

Train on 64 images for 20 steps on CPU to ensure the loss goes down before a full training run.

In [5]:
torch.manual_seed(0)
smoke_model = SmallCNNGrid().to(device)
smoke_opt = torch.optim.AdamW(smoke_model.parameters(), lr=LEARNING_RATE)
smoke_loader = DataLoader(
    torch.utils.data.Subset(train_sets['all'], range(64)),
    batch_size=16, shuffle=True,
)
smoke_model.train()
for step in range(20):
    for img, obj, dist in smoke_loader:
        img = img.to(device)
        obj = obj.to(device)
        dist = dist.to(device)
        logits, box_out = smoke_model(img)
        loss, obj_l, box_l = grid_loss(logits, box_out, obj, dist)
        smoke_opt.zero_grad()
        loss.backward()
        smoke_opt.step()
    if step % 4 == 0:
        print(f"step {step:2d}  loss={loss.item():.4f}  obj={obj_l.item():.4f}  box={box_l.item():.4f}")
del smoke_model, smoke_opt, smoke_loader

step  0  loss=1.2536  obj=0.5798  box=0.6737
step  4  loss=0.7241  obj=0.2727  box=0.4515
step  8  loss=0.6751  obj=0.2878  box=0.3873
step 12  loss=0.5051  obj=0.1795  box=0.3256
step 16  loss=0.4499  obj=0.1447  box=0.3052


## 4. Training

One run per `(train_set, seed)` pair. Each run:
- seeds Python, numpy and PyTorch so shuffling and init repeat
- trains for up to `MAX_EPOCHS` with early stopping on val cell AUC (patience `PATIENCE`)
- saves `best_val_auc_model.pt` when val AUC improves
- saves `checkpoint_epoch_XX.pt` every 5 epochs and resumes from the latest one on rerun
- skips the run entirely if its folder already has `val_cell_predictions.npz`

Val cell AUC is `roc_auc_score` over every val cell with `objectness != -1`, i.e. on positive vs. hard-negative cells only, mirroring the objectness loss.

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    obj_losses, box_losses = [], []
    for img, obj, dist in loader:
        img = img.to(device)
        obj = obj.to(device)
        dist = dist.to(device)
        logits, box_out = model(img)
        loss, obj_l, box_l = grid_loss(logits, box_out, obj, dist)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        obj_losses.append(obj_l.item())
        box_losses.append(box_l.item())
    return float(np.mean(obj_losses)), float(np.mean(box_losses))

def evaluate(model, loader):
    # Returns (mean obj loss, mean box loss, scores (n, 32, 16), boxes (n, 32, 16, 4),
    #          objectness (n, 32, 16)) with no grad.
    model.eval()
    obj_losses, box_losses = [], []
    all_scores, all_boxes, all_obj = [], [], []
    with torch.no_grad():
        for img, obj, dist in loader:
            img = img.to(device)
            obj = obj.to(device)
            dist = dist.to(device)
            logits, box_out = model(img)
            _, obj_l, box_l = grid_loss(logits, box_out, obj, dist)
            obj_losses.append(obj_l.item())
            box_losses.append(box_l.item())
            all_scores.append(torch.sigmoid(logits).cpu().numpy())
            all_boxes.append(box_out.cpu().numpy())
            all_obj.append(obj.cpu().numpy())
    scores = np.concatenate(all_scores, axis=0)
    boxes = np.concatenate(all_boxes, axis=0)
    objectness = np.concatenate(all_obj, axis=0)
    return float(np.mean(obj_losses)), float(np.mean(box_losses)), scores, boxes, objectness

def val_cell_auc(scores, objectness):
    # ROC-AUC over cells with objectness != -1
    valid = objectness != -1
    y_true = (objectness[valid] == 1).astype(np.int64)
    y_score = scores[valid]
    if y_true.min() == y_true.max():
        return float('nan')
    return float(roc_auc_score(y_true, y_score))

def train_run(train_set, seed):
    run_name = f"small-cnn_{train_set}_seed{seed}"
    run_dir = NOTEBOOK_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    # Skip if predictions already exist
    if (run_dir / 'val_cell_predictions.npz').exists():
        print(f"Skipping {run_name}: val_cell_predictions.npz already exists")
        return run_dir

    print(f"\n=== {run_name} ===")
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Build the model and optimizer after seeding so init repeats
    model = SmallCNNGrid().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(
        train_sets[train_set], batch_size=BATCH_SIZE, shuffle=True, generator=g,
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    # Resume from the latest checkpoint if one exists
    history = []
    start_epoch = 0
    best_auc = -1.0
    epochs_since_best = 0
    checkpoints = sorted(run_dir.glob('checkpoint_epoch_*.pt'))
    if checkpoints:
        latest = checkpoints[-1]
        print(f"Resuming from {latest.name}")
        state = torch.load(latest, map_location=device)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        start_epoch = state['epoch'] + 1
        history = state['history']
        best_auc = state.get('best_auc', -1.0)
        epochs_since_best = state.get('epochs_since_best', 0)

    # Time the first epoch
    first_epoch_time = None

    for epoch in range(start_epoch, MAX_EPOCHS):
        t0 = time.perf_counter()
        tr_obj, tr_box = train_one_epoch(model, train_loader, optimizer)
        va_obj, va_box, va_scores, _, va_objectness = evaluate(model, val_loader)
        va_auc = val_cell_auc(va_scores, va_objectness)
        dt = time.perf_counter() - t0
        if first_epoch_time is None:
            first_epoch_time = dt

        history.append({
            'epoch': epoch,
            'train_obj_loss': tr_obj, 'train_box_loss': tr_box,
            'val_obj_loss': va_obj, 'val_box_loss': va_box,
            'val_cell_auc': va_auc, 'seconds': dt,
        })
        print(
            f"ep {epoch:02d}  tr_obj={tr_obj:.4f} tr_box={tr_box:.4f}  "
            f"va_obj={va_obj:.4f} va_box={va_box:.4f}  val_auc={va_auc:.4f}  ({dt:.1f}s)"
        )

        # Best-model save
        if va_auc > best_auc:
            best_auc = va_auc
            epochs_since_best = 0
            torch.save(model.state_dict(), run_dir / 'best_val_auc_model.pt')
        else:
            epochs_since_best += 1

        # Periodic checkpoint
        if (epoch + 1) % 5 == 0:
            torch.save({
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'history': history,
                'best_auc': best_auc,
                'epochs_since_best': epochs_since_best,
            }, run_dir / f"checkpoint_epoch_{epoch:02d}.pt")

        # Save history every epoch so a crash still leaves a usable CSV
        pd.DataFrame(history).to_csv(run_dir / 'training_history.csv', index=False)

        if epochs_since_best >= PATIENCE:
            print(f"Early stop at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

    print(f"Best val cell AUC: {best_auc:.4f}   first epoch: {first_epoch_time:.1f}s")
    return run_dir

# Train every (seed, train set) pair. Already-finished runs skip themselves.
run_dirs = []
for seed in SEEDS:
    for train_set in ['all', 'clean', 'all-castpairs']:
        run_dirs.append(train_run(train_set, seed))


=== small-cnn_all_seed42 ===
ep 00  tr_obj=0.2257 tr_box=0.4521  va_obj=0.1663 va_box=0.4078  val_auc=0.9761  (14546.8s)
ep 01  tr_obj=0.1418 tr_box=0.3814  va_obj=0.2286 va_box=0.3993  val_auc=0.9802  (4297.6s)
ep 02  tr_obj=0.1209 tr_box=0.3574  va_obj=0.1343 va_box=0.3653  val_auc=0.9860  (2449.1s)
ep 03  tr_obj=0.1083 tr_box=0.3419  va_obj=0.1228 va_box=0.3635  val_auc=0.9870  (2554.0s)
ep 04  tr_obj=0.0991 tr_box=0.3309  va_obj=0.1193 va_box=0.3469  val_auc=0.9879  (2426.5s)
ep 05  tr_obj=0.0935 tr_box=0.3231  va_obj=0.1366 va_box=0.3425  val_auc=0.9872  (2621.6s)
ep 06  tr_obj=0.0876 tr_box=0.3148  va_obj=0.1151 va_box=0.3369  val_auc=0.9895  (2593.0s)
ep 07  tr_obj=0.0837 tr_box=0.3085  va_obj=0.1076 va_box=0.3436  val_auc=0.9910  (2686.8s)
ep 08  tr_obj=0.0796 tr_box=0.3021  va_obj=0.0985 va_box=0.3312  val_auc=0.9910  (2613.9s)
ep 09  tr_obj=0.0768 tr_box=0.2976  va_obj=0.1127 va_box=0.3424  val_auc=0.9901  (2426.5s)
ep 10  tr_obj=0.0738 tr_box=0.2916  va_obj=0.1048 va_box=0.

## 5. Predict and save

Load `best_val_auc_model.pt` per run. For val and test, write `val_cell_predictions.npz` and `test_cell_predictions.npz` in the format Fahim's scoring notebook expects:
- `rows`: int row numbers in `graz.csv`
- `scores`: `(n, 32, 16)` fracture probability per cell
- `distances`: `(n, 32, 16, 4)` predicted `l, t, r, b` in pixels, clipped to `[1, 400]`

In [ ]:
def save_predictions(run_dir):
    # If already saved, skip
    if (run_dir / 'val_cell_predictions.npz').exists() and (run_dir / 'test_cell_predictions.npz').exists():
        print(f"{run_dir.name}: predictions already saved")
        return

    model = SmallCNNGrid().to(device)
    model.load_state_dict(torch.load(run_dir / 'best_val_auc_model.pt', map_location=device))
    model.eval()

    for split_name, split_rows, split_ds in [
        ('val', val_rows, val_ds),
        ('test', test_rows, test_ds),
    ]:
        loader = DataLoader(split_ds, batch_size=BATCH_SIZE, shuffle=False)
        all_scores, all_boxes = [], []
        with torch.no_grad():
            for img, _, _ in loader:
                img = img.to(device)
                logits, box_out = model(img)
                all_scores.append(torch.sigmoid(logits).cpu().numpy())
                all_boxes.append(box_out.cpu().numpy())
        scores = np.concatenate(all_scores, axis=0).astype(np.float32)
        box_out_all = np.concatenate(all_boxes, axis=0).astype(np.float32)
        # Decode: distance = CELL_SIZE * exp(box_out), clipped to [1, 400]
        distances = np.clip(CELL_SIZE * np.exp(box_out_all), 1.0, 400.0).astype(np.float32)
        out_path = run_dir / f"{split_name}_cell_predictions.npz"
        np.savez(out_path, rows=split_rows.astype(np.int64), scores=scores, distances=distances)
        print(f"  {split_name}: saved {out_path.name}  scores {scores.shape}  distances {distances.shape}")

for run_dir in run_dirs:
    print(run_dir.name)
    save_predictions(run_dir)

## 6. Learning curves, AUC table and score heatmaps

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
titles = ['Objectness loss', 'Box loss', 'Val cell AUC']

for run_dir in run_dirs:
    hist_path = run_dir / 'training_history.csv'
    if not hist_path.exists():
        continue
    hist = pd.read_csv(hist_path)
    label = run_dir.name.replace('small-cnn_', '')

    axes[0].plot(hist['epoch'], hist['train_obj_loss'], label=f"{label} train")
    axes[0].plot(hist['epoch'], hist['val_obj_loss'], linestyle='--', label=f"{label} val")

    axes[1].plot(hist['epoch'], hist['train_box_loss'], label=f"{label} train")
    axes[1].plot(hist['epoch'], hist['val_box_loss'], linestyle='--', label=f"{label} val")

    axes[2].plot(hist['epoch'], hist['val_cell_auc'], label=label)

for ax, title in zip(axes, titles):
    ax.set_title(title)
    ax.set_xlabel('epoch')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc='best')
axes[0].set_ylabel('loss')
axes[1].set_ylabel('loss')
axes[2].set_ylabel('AUC')
plt.tight_layout()
plt.show()

# Best val cell AUC per run, with time spent
rows = []
for run_dir in run_dirs:
    hist_path = run_dir / 'training_history.csv'
    if not hist_path.exists():
        continue
    hist = pd.read_csv(hist_path)
    rows.append({
        'run': run_dir.name,
        'epochs': len(hist),
        'best_val_cell_auc': hist['val_cell_auc'].max(),
        'best_epoch': int(hist['val_cell_auc'].idxmax()),
        'total_seconds': float(hist['seconds'].sum()),
        'first_epoch_seconds': float(hist['seconds'].iloc[0]),
    })
auc_table = pd.DataFrame(rows).sort_values('best_val_cell_auc', ascending=False)
display(auc_table.round({'best_val_cell_auc': 4, 'total_seconds': 1, 'first_epoch_seconds': 1}))

# Mean val score map on fractured vs normal images.
# If both heatmaps look the same, the model is finding the wrist area, not the fracture.
val_labels = df.loc[val_mask, 'label'].to_numpy()

fig, axes = plt.subplots(len(run_dirs), 2, figsize=(7, 3 * len(run_dirs)), squeeze=False)
for i, run_dir in enumerate(run_dirs):
    preds_path = run_dir / 'val_cell_predictions.npz'
    if not preds_path.exists():
        continue
    data = np.load(preds_path)
    scores = data['scores']  # (n_val, 32, 16)
    frac_mean = scores[val_labels == 1].mean(axis=0)
    norm_mean = scores[val_labels == 0].mean(axis=0)
    vmax = max(frac_mean.max(), norm_mean.max())

    for ax, heat, name in [(axes[i, 0], frac_mean, 'fractured'),
                           (axes[i, 1], norm_mean, 'normal')]:
        im = ax.imshow(heat, vmin=0, vmax=vmax, aspect='auto', cmap='magma')
        ax.set_title(f"{run_dir.name.replace('small-cnn_', '')} - {name}")
        ax.set_xlabel('cell col')
        ax.set_ylabel('cell row')
        plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()